# Task 4: 实验对比

本Notebook完成以下任务：
1. 对比不同的基础模型（DistilBERT, RoBERTa, BERT）
2. 对比不同的超参数设置
3. 记录所有实验结果到Excel文件

## 4.1 导入必要的库

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn openpyxl

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import time
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer
)
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import os

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

## 4.2 加载数据

In [ ]:
# 加载Pipeline 1数据
with open('GroupXX_Dataset_files/train_aspect_detection.json', 'r') as f:
    train_aspect = json.load(f)
with open('GroupXX_Dataset_files/trial_aspect_detection.json', 'r') as f:
    val_aspect = json.load(f)

# 加载Pipeline 2数据
with open('GroupXX_Dataset_files/train_sentiment.json', 'r') as f:
    train_sentiment = json.load(f)
with open('GroupXX_Dataset_files/trial_sentiment.json', 'r') as f:
    val_sentiment = json.load(f)

ASPECT_CATEGORIES = ['food', 'service', 'price', 'ambience', 'anecdotes/miscellaneous']
SENTIMENT_LABELS = ['positive', 'negative', 'neutral', 'conflict']

print(f"Pipeline 1 - 训练: {len(train_aspect)}, 验证: {len(val_aspect)}")
print(f"Pipeline 2 - 训练: {len(train_sentiment)}, 验证: {len(val_sentiment)}")

## 4.3 定义实验函数

In [ ]:
def prepare_aspect_dataset(data):
    """准备Pipeline 1数据集"""
    return Dataset.from_dict({
        'text': [item['text'] for item in data],
        'labels': [item['labels'] for item in data]
    })

def prepare_sentiment_dataset(data):
    """准备Pipeline 2数据集"""
    processed = []
    for item in data:
        text = f"[ASPECT] {item['aspect']} [TEXT] {item['text']}"
        processed.append({'text': text, 'labels': item['sentiment_id']})
    return Dataset.from_dict({
        'text': [p['text'] for p in processed],
        'labels': [p['labels'] for p in processed]
    })

# 准备数据集
train_aspect_ds = prepare_aspect_dataset(train_aspect)
val_aspect_ds = prepare_aspect_dataset(val_aspect)
train_sentiment_ds = prepare_sentiment_dataset(train_sentiment)
val_sentiment_ds = prepare_sentiment_dataset(val_sentiment)

print("✓ 数据集准备完成")

In [ ]:
def compute_metrics_aspect(eval_pred):
    """Pipeline 1评估指标"""
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)
    return {
        'f1_micro': f1_score(labels, predictions, average='micro'),
        'precision_micro': precision_score(labels, predictions, average='micro', zero_division=0),
        'recall_micro': recall_score(labels, predictions, average='micro', zero_division=0)
    }

def compute_metrics_sentiment(eval_pred):
    """Pipeline 2评估指标"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

print("✓ 评估函数定义完成")

## 4.4 模型对比实验

In [ ]:
# 定义要对比的模型
MODELS_TO_COMPARE = [
    {
        "name": "distilbert-base-uncased",
        "desc": "DistilBERT (轻量级BERT)",
    },
    {
        "name": "bert-base-uncased",
        "desc": "BERT Base",
    },
    {
        "name": "roberta-base",
        "desc": "RoBERTa Base",
    }
]

print("将对比以下模型:")
for i, m in enumerate(MODELS_TO_COMPARE, 1):
    print(f"{i}. {m['desc']} ({m['name']})")

In [ ]:
def run_experiment(model_name, task_type, train_ds, val_ds, epochs=3, lr=2e-5, bs=16):
    """
    运行单个实验
    
    Args:
        model_name: 预训练模型名称
        task_type: 'aspect' 或 'sentiment'
        train_ds: 训练数据集
        val_ds: 验证数据集
        epochs: 训练轮数
        lr: 学习率
        bs: batch size
    
    Returns:
        实验结果字典
    """
    print(f"\n===== 运行实验 =====")
    print(f"模型: {model_name}")
    print(f"任务: {task_type}")
    
    start_time = time.time()
    
    try:
        # 加载tokenizer和模型
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if task_type == 'aspect':
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, 
                num_labels=5,
                problem_type="multi_label_classification"
            ).to(device)
            compute_metrics = compute_metrics_aspect
            max_length = 128
        else:
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, 
                num_labels=4
            ).to(device)
            compute_metrics = compute_metrics_sentiment
            max_length = 256
        
        # Tokenize
        def tokenize_fn(examples):
            return tokenizer(
                examples["text"],
                padding="max_length",
                truncation=True,
                max_length=max_length
            )
        
        train_tok = train_ds.map(tokenize_fn, batched=True)
        val_tok = val_ds.map(tokenize_fn, batched=True)
        
        train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        
        # 训练参数
        output_dir = f"./exp_{task_type}_{model_name.replace('/', '_')}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=lr,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs,
            num_train_epochs=epochs,
            weight_decay=0.01,
            load_best_model_at_end=True,
            logging_dir=f"{output_dir}/logs",
            logging_steps=100,
            save_total_limit=1,
            disable_tqdm=False
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_tok,
            eval_dataset=val_tok,
            compute_metrics=compute_metrics
        )
        
        # 训练
        trainer.train()
        
        # 评估
        eval_results = trainer.evaluate()
        
        elapsed_time = time.time() - start_time
        
        # 清理
        import shutil
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        
        return {
            'model': model_name,
            'task': task_type,
            'epochs': epochs,
            'learning_rate': lr,
            'batch_size': bs,
            'eval_results': eval_results,
            'elapsed_time': elapsed_time
        }
    
    except Exception as e:
        print(f"错误: {e}")
        return {
            'model': model_name,
            'task': task_type,
            'error': str(e)
        }

## 4.5 运行Pipeline 1实验（方面类别检测）

In [ ]:
# 运行Pipeline 1实验（使用少量epoch进行快速对比）
print("===== Pipeline 1: 方面类别检测实验 =====")

results_pipeline1 = []

for model_info in MODELS_TO_COMPARE:
    result = run_experiment(
        model_name=model_info['name'],
        task_type='aspect',
        train_ds=train_aspect_ds,
        val_ds=val_aspect_ds,
        epochs=3,
        lr=2e-5,
        bs=16
    )
    results_pipeline1.append(result)
    
    if 'error' not in result:
        print(f"\n✓ {model_info['desc']} 完成")
        print(f"  F1 (micro): {result['eval_results'].get('eval_f1_micro', 0):.4f}")
        print(f"  时间: {result['elapsed_time']:.2f}秒")

## 4.6 运行Pipeline 2实验（方面情感分析）

In [ ]:
# 运行Pipeline 2实验
print("\n===== Pipeline 2: 方面情感分析实验 =====")

results_pipeline2 = []

for model_info in MODELS_TO_COMPARE:
    result = run_experiment(
        model_name=model_info['name'],
        task_type='sentiment',
        train_ds=train_sentiment_ds,
        val_ds=val_sentiment_ds,
        epochs=3,
        lr=2e-5,
        bs=32
    )
    results_pipeline2.append(result)
    
    if 'error' not in result:
        print(f"\n✓ {model_info['desc']} 完成")
        print(f"  准确率: {result['eval_results'].get('eval_accuracy', 0):.4f}")
        print(f"  F1 (weighted): {result['eval_results'].get('eval_f1_weighted', 0):.4f}")
        print(f"  时间: {result['elapsed_time']:.2f}秒")

## 4.7 超参数对比实验

In [ ]:
# 对比不同学习率
print("\n===== 超参数对比: 学习率 =====")

learning_rates = [1e-5, 2e-5, 5e-5]
results_lr = []

for lr in learning_rates:
    result = run_experiment(
        model_name="distilbert-base-uncased",
        task_type='aspect',
        train_ds=train_aspect_ds,
        val_ds=val_aspect_ds,
        epochs=3,
        lr=lr,
        bs=16
    )
    results_lr.append(result)
    
    if 'error' not in result:
        print(f"\n学习率 {lr}: F1={result['eval_results'].get('eval_f1_micro', 0):.4f}")

In [ ]:
# 对比不同batch size
print("\n===== 超参数对比: Batch Size =====")

batch_sizes = [8, 16, 32]
results_bs = []

for bs in batch_sizes:
    result = run_experiment(
        model_name="distilbert-base-uncased",
        task_type='aspect',
        train_ds=train_aspect_ds,
        val_ds=val_aspect_ds,
        epochs=3,
        lr=2e-5,
        bs=bs
    )
    results_bs.append(result)
    
    if 'error' not in result:
        print(f"\nBatch Size {bs}: F1={result['eval_results'].get('eval_f1_micro', 0):.4f}")

## 4.8 生成实验结果表格

In [ ]:
# 整理Pipeline 1结果
rows1 = []
for r in results_pipeline1:
    if 'error' not in r:
        rows1.append({
            'Experiment': f"Model Comparison - {r['model']}",
            'Pipeline': 'Aspect Detection',
            'Model': r['model'],
            'Learning Rate': r['learning_rate'],
            'Batch Size': r['batch_size'],
            'Epochs': r['epochs'],
            'F1 (micro)': r['eval_results'].get('eval_f1_micro', 0),
            'Precision (micro)': r['eval_results'].get('eval_precision_micro', 0),
            'Recall (micro)': r['eval_results'].get('eval_recall_micro', 0),
            'Time (s)': r['elapsed_time']
        })

df1 = pd.DataFrame(rows1)
print("Pipeline 1 模型对比结果:")
print(df1)

In [ ]:
# 整理Pipeline 2结果
rows2 = []
for r in results_pipeline2:
    if 'error' not in r:
        rows2.append({
            'Experiment': f"Model Comparison - {r['model']}",
            'Pipeline': 'Sentiment Analysis',
            'Model': r['model'],
            'Learning Rate': r['learning_rate'],
            'Batch Size': r['batch_size'],
            'Epochs': r['epochs'],
            'Accuracy': r['eval_results'].get('eval_accuracy', 0),
            'F1 (weighted)': r['eval_results'].get('eval_f1_weighted', 0),
            'Time (s)': r['elapsed_time']
        })

df2 = pd.DataFrame(rows2)
print("\nPipeline 2 模型对比结果:")
print(df2)

In [ ]:
# 整理学习率实验结果
rows_lr = []
for r in results_lr:
    if 'error' not in r:
        rows_lr.append({
            'Experiment': 'Learning Rate Comparison',
            'Pipeline': 'Aspect Detection',
            'Model': r['model'],
            'Learning Rate': r['learning_rate'],
            'Batch Size': r['batch_size'],
            'Epochs': r['epochs'],
            'F1 (micro)': r['eval_results'].get('eval_f1_micro', 0),
            'Precision (micro)': r['eval_results'].get('eval_precision_micro', 0),
            'Recall (micro)': r['eval_results'].get('eval_recall_micro', 0),
            'Time (s)': r['elapsed_time']
        })

df_lr = pd.DataFrame(rows_lr)
print("\n学习率对比结果:")
print(df_lr)

In [ ]:
# 整理Batch Size实验结果
rows_bs = []
for r in results_bs:
    if 'error' not in r:
        rows_bs.append({
            'Experiment': 'Batch Size Comparison',
            'Pipeline': 'Aspect Detection',
            'Model': r['model'],
            'Learning Rate': r['learning_rate'],
            'Batch Size': r['batch_size'],
            'Epochs': r['epochs'],
            'F1 (micro)': r['eval_results'].get('eval_f1_micro', 0),
            'Precision (micro)': r['eval_results'].get('eval_precision_micro', 0),
            'Recall (micro)': r['eval_results'].get('eval_recall_micro', 0),
            'Time (s)': r['elapsed_time']
        })

df_bs = pd.DataFrame(rows_bs)
print("\nBatch Size对比结果:")
print(df_bs)

## 4.9 保存到Excel文件

In [ ]:
# 创建Excel文件
excel_path = 'GroupXX_documentation/Experimental_results.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df1.to_excel(writer, sheet_name='Pipeline1_Model_Comparison', index=False)
    df2.to_excel(writer, sheet_name='Pipeline2_Model_Comparison', index=False)
    df_lr.to_excel(writer, sheet_name='Learning_Rate_Comparison', index=False)
    df_bs.to_excel(writer, sheet_name='Batch_Size_Comparison', index=False)
    
    # 创建汇总表
    summary_data = {
        '实验类型': ['模型对比', '模型对比', '模型对比', '超参数对比', '超参数对比'],
        'Pipeline': ['Aspect Detection', 'Sentiment Analysis', 'Aspect Detection', 'Aspect Detection', 'Aspect Detection'],
        '变量': ['Model', 'Model', 'Model', 'Learning Rate', 'Batch Size'],
        '测试值数量': [3, 3, 3, 3, 3],
        '最佳配置': [
            df1.loc[df1['F1 (micro)'].idxmax(), 'Model'] if len(df1) > 0 else 'N/A',
            df2.loc[df2['Accuracy'].idxmax(), 'Model'] if len(df2) > 0 else 'N/A',
            'distilbert-base-uncased',
            f"lr={df_lr.loc[df_lr['F1 (micro)'].idxmax(), 'Learning Rate']}" if len(df_lr) > 0 else 'N/A',
            f"bs={df_bs.loc[df_bs['F1 (micro)'].idxmax(), 'Batch Size']}" if len(df_bs) > 0 else 'N/A'
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"✓ Excel文件已保存: {excel_path}")

## 4.10 实验结论

In [ ]:
print("\n===== 实验对比结论 =====\n")

print("1. 模型选择:")
if len(df1) > 0:
    best_model_p1 = df1.loc[df1['F1 (micro)'].idxmax()]
    print(f"   - Pipeline 1最佳模型: {best_model_p1['Model']} (F1={best_model_p1['F1 (micro)']:.4f})")
if len(df2) > 0:
    best_model_p2 = df2.loc[df2['Accuracy'].idxmax()]
    print(f"   - Pipeline 2最佳模型: {best_model_p2['Model']} (Acc={best_model_p2['Accuracy']:.4f})")

print("\n2. 超参数选择:")
if len(df_lr) > 0:
    best_lr = df_lr.loc[df_lr['F1 (micro)'].idxmax()]
    print(f"   - 最佳学习率: {best_lr['Learning Rate']} (F1={best_lr['F1 (micro)']:.4f})")
if len(df_bs) > 0:
    best_bs = df_bs.loc[df_bs['F1 (micro)'].idxmax()]
    print(f"   - 最佳Batch Size: {best_bs['Batch Size']} (F1={best_bs['F1 (micro)']:.4f})")

print("\n3. 效率对比:")
if len(df1) > 0:
    fastest = df1.loc[df1['Time (s)'].idxmin()]
    print(f"   - Pipeline 1最快模型: {fastest['Model']} ({fastest['Time (s)']:.2f}秒)")

print("\n✓ 实验对比完成！")